# Test Tracking & Results Storage

Easy access to keep track of what iterations were run + their corresponding results.

In [57]:
import dask.dataframe as dd
import pandas as pd
import numpy as np
from pathlib import Path
import glob

Old test b4 implementing parallel dask processing, done on Kevin Mac.
```bash
#!/bin/bash

python pivot_all_files.py \
    --input-dir "s3://dsc291-ucsd/taxi/Dataset/2023/yellow_taxi/" \
    --output-dir data/remote_test_output/ \
```
```
INFO:__main__:Discovered 9 Parquet files
INFO:__main__:Processing 2023-01 (1 files)
INFO:__main__:Processing 2023-02 (1 files)
INFO:__main__:Processing 2023-03 (1 files)
INFO:__main__:Processing 2023-04 (1 files)
INFO:__main__:Processing 2023-05 (1 files)
INFO:__main__:Processing 2023-06 (1 files)
INFO:__main__:Processing 2023-07 (1 files)
INFO:__main__:Processing 2023-08 (1 files)
INFO:__main__:Processing 2023-09 (1 files)
INFO:__main__:Pipeline complete
INFO:__main__:input_rows: 28071659
INFO:__main__:bad_parse_rows: 0
INFO:__main__:month_mismatch_rows: 603
INFO:__main__:rows_dropped_low_count: 41701
INFO:__main__:rows_kept: 19033
INFO:__main__:output_rows: 19033
INFO:__main__:final_output_rows: 19033
INFO:__main__:runtime_seconds: 156
```

In [3]:
taxi_wide_df = pd.read_parquet(
    "./data/remote_test_output/taxi_wide_table.parquet"
)

pd.set_option('display.max_columns', None)
taxi_wide_df

,taxi_type,date,pickup_place,hour_0,hour_1,hour_2,hour_3,hour_4,hour_5,hour_6,hour_7,hour_8,hour_9,hour_10,hour_11,hour_12,hour_13,hour_14,hour_15,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,hour_23
0,yellow,2023-01-01,4,19.0,28.0,43.0,33.0,12.0,3.0,2.0,1.0,1.0,1.0,2.0,1.0,3.0,2.0,2.0,2.0,4.0,2.0,5.0,1.0,1.0,3.0,0.0,3.0
1,yellow,2023-01-01,7,3.0,16.0,28.0,21.0,12.0,5.0,4.0,2.0,2.0,3.0,7.0,1.0,4.0,1.0,0.0,3.0,5.0,3.0,0.0,2.0,1.0,2.0,0.0,1.0
2,yellow,2023-01-01,13,14.0,18.0,11.0,7.0,6.0,2.0,1.0,3.0,6.0,10.0,19.0,30.0,40.0,48.0,46.0,33.0,38.0,30.0,13.0,13.0,5.0,2.0,3.0,0.0
3,yellow,2023-01-01,24,20.0,12.0,13.0,5.0,11.0,2.0,5.0,2.0,7.0,12.0,14.0,7.0,12.0,13.0,8.0,10.0,17.0,14.0,14.0,8.0,6.0,5.0,2.0,1.0
4,yellow,2023-01-01,33,12.0,9.0,7.0,3.0,1.0,0.0,0.0,0.0,2.0,3.0,3.0,1.0,4.0,3.0,3.0,2.0,4.0,4.0,3.0,1.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19028,yellow,2023-09-30,261,10.0,4.0,0.0,1.0,4.0,1.0,4.0,2.0,11.0,20.0,46.0,40.0,57.0,51.0,46.0,61.0,78.0,81.0,56.0,51.0,24.0,26.0,37.0,42.0
19029,yellow,2023-09-30,262,12.0,2.0,3.0,4.0,5.0,2.0,27.0,23.0,76.0,91.0,127.0,125.0,74.0,90.0,101.0,65.0,63.0,72.0,96.0,101.0,47.0,38.0,23.0,26.0
19030,yellow,2023-09-30,263,56.0,38.0,29.0,22.0,19.0,6.0,21.0,44.0,79.0,96.0,127.0,168.0,126.0,132.0,116.0,113.0,93.0,130.0,161.0,170.0,129.0,98.0,106.0,116.0
19031,yellow,2023-09-30,264,54.0,32.0,31.0,30.0,13.0,5.0,14.0,15.0,31.0,34.0,66.0,56.0,57.0,63.0,59.0,64.0,78.0,82.0,82.0,92.0,70.0,65.0,63.0,72.0


First test of parallel implentation, run on Kevin PC.
```bash
python pivot_all_files.py \
    --input-dir "s3://dsc291-ucsd/taxi/Dataset/2023/yellow_taxi/" \
    --output-dir data/remote_test_output/ \
    --workers 8
```
```
INFO:__main__:Using 8 Dask workers
INFO:__main__:Discovered 9 Parquet files
INFO:__main__:Processing 2023-01 (1 files)
INFO:__main__:Optimal partition size for 2023-01: 134217728B
INFO:__main__:Processing 2023-02 (1 files)
INFO:__main__:Optimal partition size for 2023-02: 268435456B
INFO:__main__:Processing 2023-03 (1 files)
INFO:__main__:Optimal partition size for 2023-03: 268435456B
INFO:__main__:Processing 2023-04 (1 files)
INFO:__main__:Optimal partition size for 2023-04: 268435456B
INFO:__main__:Processing 2023-05 (1 files)
INFO:__main__:Optimal partition size for 2023-05: 268435456B
INFO:__main__:Processing 2023-06 (1 files)
INFO:__main__:Optimal partition size for 2023-06: 268435456B
INFO:__main__:Processing 2023-07 (1 files)
INFO:__main__:Optimal partition size for 2023-07: 268435456B
INFO:__main__:Processing 2023-08 (1 files)
INFO:__main__:Optimal partition size for 2023-08: 268435456B
INFO:__main__:Processing 2023-09 (1 files)
INFO:__main__:Optimal partition size for 2023-09: 134217728B
INFO:__main__:Pipeline complete
INFO:__main__:input_rows: 28071659
INFO:__main__:bad_parse_rows: 0
INFO:__main__:month_mismatch_rows: 603
INFO:__main__:rows_dropped_low_count: 41701
INFO:__main__:rows_kept: 19033
INFO:__main__:output_rows: 19033
INFO:__main__:final_output_rows: 19033
INFO:__main__:runtime_seconds: 73
```

Single worker test for compute diff
```bash
python pivot_all_files.py \
    --input-dir "s3://dsc291-ucsd/taxi/Dataset/2023/yellow_taxi/" \
    --output-dir data/remote_test_output_dask_parallel/ \
    --workers 1
```
```
INFO:__main__:Using 1 Dask workers
INFO:__main__:Discovered 9 Parquet files
INFO:__main__:Processing 2023-01 (1 files)
INFO:__main__:Optimal partition size for 2023-01: 134217728B
INFO:__main__:Processing 2023-02 (1 files)
INFO:__main__:Optimal partition size for 2023-02: 268435456B
INFO:__main__:Processing 2023-03 (1 files)
INFO:__main__:Optimal partition size for 2023-03: 268435456B
INFO:__main__:Processing 2023-04 (1 files)
INFO:__main__:Optimal partition size for 2023-04: 268435456B
INFO:__main__:Processing 2023-05 (1 files)
INFO:__main__:Optimal partition size for 2023-05: 268435456B
INFO:__main__:Processing 2023-06 (1 files)
INFO:__main__:Optimal partition size for 2023-06: 268435456B
INFO:__main__:Processing 2023-07 (1 files)
INFO:__main__:Optimal partition size for 2023-07: 268435456B
INFO:__main__:Processing 2023-08 (1 files)
INFO:__main__:Optimal partition size for 2023-08: 134217728B
INFO:__main__:Processing 2023-09 (1 files)
INFO:__main__:Optimal partition size for 2023-09: 134217728B
INFO:__main__:Pipeline complete
INFO:__main__:input_rows: 28071659
INFO:__main__:bad_parse_rows: 0
INFO:__main__:month_mismatch_rows: 603
INFO:__main__:rows_dropped_low_count: 41701
INFO:__main__:rows_kept: 19033
INFO:__main__:output_rows: 19033
INFO:__main__:final_output_rows: 19033
INFO:__main__:runtime_seconds: 74
```


First test of new month-wise parallel implementation on Kevin PC.
```bash
python pivot_all_files.py \
    --input-dir "s3://dsc291-ucsd/taxi/Dataset/2023/yellow_taxi/" \
    --output-dir data/remote_test_output_dask_parallel/ \
    --workers 1
```
```
INFO:__main__:Using single-threaded Dask
INFO:__main__:Discovered 9 Parquet files
INFO:__main__:Scheduling 2023-01 (1 files)
INFO:__main__:Optimal partition size for 2023-01: 134217728B
INFO:__main__:Scheduling 2023-02 (1 files)
INFO:__main__:Optimal partition size for 2023-02: 134217728B
INFO:__main__:Scheduling 2023-03 (1 files)
INFO:__main__:Optimal partition size for 2023-03: 268435456B
INFO:__main__:Scheduling 2023-04 (1 files)
INFO:__main__:Optimal partition size for 2023-04: 268435456B
INFO:__main__:Scheduling 2023-05 (1 files)
INFO:__main__:Optimal partition size for 2023-05: 268435456B
INFO:__main__:Scheduling 2023-06 (1 files)
INFO:__main__:Optimal partition size for 2023-06: 134217728B
INFO:__main__:Scheduling 2023-07 (1 files)
INFO:__main__:Optimal partition size for 2023-07: 134217728B
INFO:__main__:Scheduling 2023-08 (1 files)
INFO:__main__:Optimal partition size for 2023-08: 268435456B
INFO:__main__:Scheduling 2023-09 (1 files)
INFO:__main__:Optimal partition size for 2023-09: 268435456B
INFO:__main__:Pipeline complete
INFO:__main__:input_rows: 28071659
INFO:__main__:bad_parse_rows: 0
INFO:__main__:month_mismatch_rows: 603
INFO:__main__:rows_dropped_low_count: 41701
INFO:__main__:rows_kept: 19033
INFO:__main__:output_rows: 19033
INFO:__main__:final_output_rows: 19033
INFO:__main__:runtime_seconds: 41
```

In [ ]:
# read parquet from 
# s3://dsc291-taxi/taxi-output/taxi_wide_table.parquet/

taxi_df = dd.read_parquet(
    "s3://dsc291-taxi/taxi-output/taxi_wide_table.parquet/*.parquet",
    engine="pyarrow",
    storage_options={'anon': True}
)

pd.set_option('display.max_columns', None)
taxi_df

FileNotFoundError: dsc291-taxi/taxi-output/taxi_wide_table.parquet/*.parquet

In [1]:
from io_utils import discover_parquet_files

files = discover_parquet_files("s3://dsc291-ucsd/taxi/Dataset/")
files

['s3://dsc291-ucsd/taxi/Dataset/2009/yellow_taxi/yellow_tripdata_2009-01.parquet',
 's3://dsc291-ucsd/taxi/Dataset/2009/yellow_taxi/yellow_tripdata_2009-02.parquet',
 's3://dsc291-ucsd/taxi/Dataset/2009/yellow_taxi/yellow_tripdata_2009-03.parquet',
 's3://dsc291-ucsd/taxi/Dataset/2009/yellow_taxi/yellow_tripdata_2009-04.parquet',
 's3://dsc291-ucsd/taxi/Dataset/2009/yellow_taxi/yellow_tripdata_2009-05.parquet',
 's3://dsc291-ucsd/taxi/Dataset/2009/yellow_taxi/yellow_tripdata_2009-06.parquet',
 's3://dsc291-ucsd/taxi/Dataset/2009/yellow_taxi/yellow_tripdata_2009-07.parquet',
 's3://dsc291-ucsd/taxi/Dataset/2009/yellow_taxi/yellow_tripdata_2009-08.parquet',
 's3://dsc291-ucsd/taxi/Dataset/2009/yellow_taxi/yellow_tripdata_2009-09.parquet',
 's3://dsc291-ucsd/taxi/Dataset/2009/yellow_taxi/yellow_tripdata_2009-10.parquet',
 's3://dsc291-ucsd/taxi/Dataset/2009/yellow_taxi/yellow_tripdata_2009-11.parquet',
 's3://dsc291-ucsd/taxi/Dataset/2009/yellow_taxi/yellow_tripdata_2009-12.parquet',
 's3

In [2]:
import pyarrow.parquet as pq
from tqdm import tqdm
import fsspec

schemas = []
for file in tqdm(files):
    schema = pq.read_schema(file, filesystem=fsspec.filesystem("s3", anon=True))
    schemas.append(schema)
schemas

100%|██████████| 443/443 [01:02<00:00,  7.12it/s]


[vendor_name: string
 Trip_Pickup_DateTime: string
 Trip_Dropoff_DateTime: string
 Passenger_Count: int64
 Trip_Distance: double
 Start_Lon: double
 Start_Lat: double
 Rate_Code: double
 store_and_forward: double
 End_Lon: double
 End_Lat: double
 Payment_Type: string
 Fare_Amt: double
 surcharge: double
 mta_tax: double
 Tip_Amt: double
 Tolls_Amt: double
 Total_Amt: double
 -- schema metadata --
 pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 2473,
 vendor_name: string
 Trip_Pickup_DateTime: string
 Trip_Dropoff_DateTime: string
 Passenger_Count: int64
 Trip_Distance: double
 Start_Lon: double
 Start_Lat: double
 Rate_Code: double
 store_and_forward: double
 End_Lon: double
 End_Lat: double
 Payment_Type: string
 Fare_Amt: double
 surcharge: double
 mta_tax: double
 Tip_Amt: double
 Tolls_Amt: double
 Total_Amt: double
 -- schema metadata --
 pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 2473,
 vendor_name: string
 Trip_Pic

In [54]:
# iterate through schemas, find all instances of columns related to pickup datetime and location

pickup_datetime_columns = set()
pickup_location_columns = set()

for schema in schemas:
    for field in schema:
        if "datetime" in field.name.lower():
            pickup_datetime_columns.add(field.name)
        if "location" in field.name.lower() or "pickup" in field.name.lower():
            pickup_location_columns.add(field.name)

In [55]:
pickup_datetime_columns

{'Trip_Dropoff_DateTime',
 'Trip_Pickup_DateTime',
 'dropOff_datetime',
 'dropoff_datetime',
 'lpep_dropoff_datetime',
 'lpep_pickup_datetime',
 'on_scene_datetime',
 'pickup_datetime',
 'request_datetime',
 'tpep_dropoff_datetime',
 'tpep_pickup_datetime'}

In [56]:
pickup_location_columns

{'DOLocationID',
 'DOlocationID',
 'PULocationID',
 'PUlocationID',
 'Trip_Pickup_DateTime',
 'lpep_pickup_datetime',
 'pickup_datetime',
 'pickup_latitude',
 'pickup_longitude',
 'tpep_pickup_datetime'}

In [12]:
# check if all schemas have "Trip_Pickup_DateTime", "lpep_pickup_datetime", "pickup_datetime", or "tpep_pickup_datetime"

pickup_candidates = {
    "Trip_Pickup_DateTime",
    "lpep_pickup_datetime",
    "pickup_datetime",
    "tpep_pickup_datetime",
}

for schema in schemas:
    columns = [field.name for field in schema]
    if not any(col in columns for col in pickup_candidates):
        print("Schema missing pickup datetime column:", schema)

In [15]:
# check if all schemas have 'pulocationid'

missing_pulocationid_schemas = []

for file, schema in zip(files, schemas):
    columns = [field.name.lower() for field in schema]
    if "pulocationid" not in columns:
        print(f"Schema missing pickup location column in file {file}:", schema)
        missing_pulocationid_schemas.append(schema)

Schema missing pickup location column in file s3://dsc291-ucsd/taxi/Dataset/2009/yellow_taxi/yellow_tripdata_2009-01.parquet: vendor_name: string
Trip_Pickup_DateTime: string
Trip_Dropoff_DateTime: string
Passenger_Count: int64
Trip_Distance: double
Start_Lon: double
Start_Lat: double
Rate_Code: double
store_and_forward: double
End_Lon: double
End_Lat: double
Payment_Type: string
Fare_Amt: double
surcharge: double
mta_tax: double
Tip_Amt: double
Tolls_Amt: double
Total_Amt: double
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 2473
Schema missing pickup location column in file s3://dsc291-ucsd/taxi/Dataset/2009/yellow_taxi/yellow_tripdata_2009-02.parquet: vendor_name: string
Trip_Pickup_DateTime: string
Trip_Dropoff_DateTime: string
Passenger_Count: int64
Trip_Distance: double
Start_Lon: double
Start_Lat: double
Rate_Code: double
store_and_forward: double
End_Lon: double
End_Lat: double
Payment_Type: string
Fare_Amt: double
surcharge:

In [17]:
# find any columns related to location in missing_pulocationid_schemas

possible_location_cols = set()

for schema in missing_pulocationid_schemas:
    for field in schema:
        for keyword in ["location", "loc", "lat", "lon"]:
            if keyword in field.name.lower():
                possible_location_cols.add(field.name)
                break

possible_location_cols

{'End_Lat',
 'End_Lon',
 'Start_Lat',
 'Start_Lon',
 'dropoff_latitude',
 'dropoff_longitude',
 'pickup_latitude',
 'pickup_longitude'}

In [38]:
# read taxi_zones.shp

import geopandas as gpd

taxi_zones = gpd.read_file("taxi_zones.shx")
taxi_zones = taxi_zones.set_crs(epsg=2263).to_crs(epsg=4326)
taxi_zones["geometry"]

0      POLYGON ((-74.18445 40.695, -74.18449 40.6951,...
1      MULTIPOLYGON (((-73.82338 40.63899, -73.82277 ...
2      POLYGON ((-73.84793 40.87134, -73.84725 40.870...
3      POLYGON ((-73.97177 40.72582, -73.97179 40.725...
4      POLYGON ((-74.17422 40.56257, -74.17349 40.562...
                             ...                        
258    POLYGON ((-73.85107 40.91037, -73.85207 40.909...
259    POLYGON ((-73.90175 40.76078, -73.90147 40.759...
260    POLYGON ((-74.01333 40.70503, -74.01327 40.704...
261    MULTIPOLYGON (((-73.94383 40.78286, -73.94376 ...
262    POLYGON ((-73.95219 40.77302, -73.95269 40.772...
Name: geometry, Length: 263, dtype: geometry

In [40]:
sindex = taxi_zones.sindex
candidates = list(sindex.intersection((0, 0)))  
candidates

[]

In [69]:
# read s3://dsc291-ucsd/taxi/Dataset/2009/yellow_taxi/yellow_tripdata_2009-01.parquet

df = pd.read_parquet(
    "s3://dsc291-ucsd/taxi/Dataset/2009/yellow_taxi/yellow_tripdata_2009-01.parquet",
    engine="pyarrow",
    storage_options={'anon': True}
)
df

,vendor_name,Trip_Pickup_DateTime,Trip_Dropoff_DateTime,Passenger_Count,Trip_Distance,Start_Lon,Start_Lat,Rate_Code,store_and_forward,End_Lon,End_Lat,Payment_Type,Fare_Amt,surcharge,mta_tax,Tip_Amt,Tolls_Amt,Total_Amt
0,VTS,2009-01-04 02:52:00,2009-01-04 03:02:00,1,2.63,-73.991957,40.721567,NaN,NaN,-73.993803,40.695922,CASH,8.9,0.5,NaN,0.00,0.0,9.40
1,VTS,2009-01-04 03:31:00,2009-01-04 03:38:00,3,4.55,-73.982102,40.736290,NaN,NaN,-73.955850,40.768030,Credit,12.1,0.5,NaN,2.00,0.0,14.60
2,VTS,2009-01-03 15:43:00,2009-01-03 15:57:00,5,10.35,-74.002587,40.739748,NaN,NaN,-73.869983,40.770225,Credit,23.7,0.0,NaN,4.74,0.0,28.44
3,DDS,2009-01-01 20:52:58,2009-01-01 21:14:00,1,5.00,-73.974267,40.790955,NaN,NaN,-73.996558,40.731849,CREDIT,14.9,0.5,NaN,3.05,0.0,18.45
4,DDS,2009-01-24 16:18:23,2009-01-24 16:24:56,1,0.40,-74.001580,40.719382,NaN,NaN,-74.008378,40.720350,CASH,3.7,0.0,NaN,0.00,0.0,3.70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14092408,VTS,2009-01-27 14:36:00,2009-01-27 14:46:00,5,0.89,-73.982013,40.743330,NaN,NaN,-73.994328,40.750130,CASH,6.5,0.0,NaN,0.00,0.0,6.50
14092409,VTS,2009-01-27 13:56:00,2009-01-27 14:02:00,1,1.94,-73.972788,40.761988,NaN,NaN,-73.951477,40.778217,Credit,8.1,0.0,NaN,1.90,0.0,10.00
14092410,CMT,2009-01-23 08:39:44,2009-01-23 09:02:15,1,3.80,-73.977467,40.751861,NaN,NaN,-74.009913,40.713470,Cash,14.5,0.0,NaN,0.00,0.0,14.50
14092411,VTS,2009-01-24 23:05:00,2009-01-24 23:15:00,3,3.85,-73.981295,40.753000,NaN,NaN,-73.949453,40.779520,CASH,10.9,0.5,NaN,0.00,0.0,11.40


In [ ]:
from shapely.geometry import Point

taxi_zones = gpd.read_file("taxi_zones.shx")
taxi_zones = taxi_zones.set_crs(epsg=2263).to_crs(epsg=4326)
sindex = taxi_zones.sindex

# vectorized spatial join using 5 mil chunk size

df_use = df.copy()
lat_col = "Start_Lat"
lon_col = "Start_Lon"
chunk_size = 5_000_000
df_use['pickup_place'] = np.nan
for start in range(0, len(df_use), chunk_size):
    end = min(start + chunk_size, len(df_use))
    chunk = df_use.iloc[start:end]

    gdf_points = gpd.GeoDataFrame(
        chunk[[lon_col, lat_col]],
        geometry=gpd.points_from_xy(chunk[lon_col], chunk[lat_col]),
        crs=taxi_zones.crs
    )

    joined = gpd.sjoin(gdf_points, taxi_zones, how="left", predicate="covers")
    df_use.loc[chunk.index, 'pickup_place'] = joined['index_right'].values + 1
df_use = df_use.drop(columns=[lat_col, lon_col])
df_use[['pickup_place']]

,pickup_place
0,148.0
1,107.0
2,249.0
3,238.0
4,144.0
...,...
14092408,170.0
14092409,163.0
14092410,170.0
14092411,164.0


In [71]:
df_use = df.copy()
lat_col = "Start_Lat"
lon_col = "Start_Lon"
chunk_size = 5_000_000
df_use['pickup_place'] = np.nan


gdf_points = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(df_use[lon_col], df_use[lat_col]),
    crs=taxi_zones.crs
)

joined = gpd.sjoin(gdf_points, taxi_zones, how="left", predicate="within")
df_use.loc[df_use.index, 'pickup_place'] = joined['index_right'].values + 1
df_use = df_use.drop(columns=[lat_col, lon_col])
df_use[['pickup_place']]

,pickup_place
0,148.0
1,107.0
2,249.0
3,238.0
4,144.0
...,...
14092408,170.0
14092409,163.0
14092410,170.0
14092411,164.0


In [44]:
from shapely.geometry import Point

# lon, lat
point = Point(-73.991957, 40.721567)
candidates = list(sindex.intersection(point.bounds))  
candidates

[np.int64(147), np.int64(78)]

In [52]:
for idx in sindex.intersection(point.bounds):
    if taxi_zones.geometry.iloc[idx].covers(point):
        print(f"Covered by zone {idx}, actual id {idx + 1}")

Covered by zone 147, actual id 148


In [50]:
print(taxi_zones.geometry[147])

POLYGON ((-73.98447731699999 40.72023423900007, -73.985073423 40.719083294000086, -73.9859109239999 40.7193389730001, -73.98675136699994 40.71959284600017, -73.98736015299995 40.718369797000136, -73.98739393199992 40.71830352300009, -73.98743405799996 40.71821308000012, -73.98751208099989 40.718072186000065, -73.98787935799997 40.7174089730001, -73.98791645999995 40.7173355740001, -73.98795181500003 40.71726562800011, -73.98836626299988 40.7164457020001, -73.98915463799979 40.716708647000075, -73.98975418199988 40.715552348000166, -73.9902023599999 40.714664430000084, -73.99022155699988 40.71457875600005, -73.99022750999988 40.714491669000125, -73.99022012699989 40.71440494000008, -73.99075515399984 40.714554073000045, -73.99122551100001 40.71448664400011, -73.99213486499995 40.71441980200008, -73.99256242199989 40.71438807700016, -73.99282173499992 40.71436523000009, -73.99301588399996 40.71435562500006, -73.99395033499991 40.71429083200014, -73.99414039099992 40.714268932000046, -73.

In [ ]:
taxi_wide_df = pd.read_parquet(
    "./data/v2_test_schema/taxi_wide_table.parquet",
    engine="pyarrow"
)

pd.set_option('display.max_columns', None)
taxi_wide_df

,taxi_type,date,pickup_place,hour_0,hour_1,hour_2,hour_3,hour_4,hour_5,hour_6,hour_7,hour_8,hour_9,hour_10,hour_11,hour_12,hour_13,hour_14,hour_15,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,hour_23
0,yellow,2009-01-01,4.0,121.0,137.0,125.0,124.0,100.0,51.0,41.0,44.0,36.0,32.0,60.0,54.0,75.0,61.0,61.0,60.0,51.0,49.0,90.0,92.0,70.0,78.0,74.0,95.0
1,yellow,2009-01-01,7.0,42.0,88.0,115.0,110.0,105.0,66.0,37.0,34.0,17.0,20.0,47.0,36.0,40.0,41.0,43.0,45.0,47.0,41.0,50.0,34.0,46.0,44.0,37.0,39.0
2,yellow,2009-01-01,12.0,4.0,4.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,3.0,5.0,27.0,39.0,45.0,36.0,42.0,21.0,8.0,3.0,2.0,1.0,6.0,0.0,1.0
3,yellow,2009-01-01,13.0,63.0,63.0,58.0,33.0,19.0,13.0,9.0,14.0,18.0,28.0,91.0,163.0,169.0,183.0,171.0,139.0,133.0,126.0,130.0,132.0,88.0,83.0,96.0,43.0
4,yellow,2009-01-01,17.0,13.0,11.0,19.0,18.0,8.0,5.0,4.0,2.0,2.0,1.0,0.0,0.0,1.0,1.0,2.0,1.0,2.0,0.0,0.0,1.0,2.0,8.0,2.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38413,yellow,2009-12-31,256.0,57.0,71.0,44.0,32.0,29.0,6.0,5.0,3.0,1.0,3.0,4.0,3.0,7.0,14.0,18.0,11.0,16.0,21.0,26.0,44.0,95.0,135.0,187.0,188.0
38414,yellow,2009-12-31,260.0,15.0,7.0,13.0,14.0,16.0,22.0,32.0,32.0,27.0,12.0,14.0,11.0,15.0,13.0,11.0,18.0,13.0,18.0,18.0,15.0,27.0,20.0,11.0,23.0
38415,yellow,2009-12-31,261.0,57.0,24.0,29.0,17.0,9.0,11.0,27.0,21.0,36.0,61.0,87.0,112.0,154.0,140.0,173.0,168.0,127.0,130.0,130.0,150.0,166.0,194.0,155.0,95.0
38416,yellow,2009-12-31,262.0,105.0,58.0,32.0,19.0,22.0,40.0,125.0,263.0,421.0,355.0,301.0,314.0,302.0,270.0,266.0,231.0,244.0,312.0,419.0,599.0,511.0,346.0,324.0,286.0


In [73]:
# data type for pickup_place
taxi_wide_df['pickup_place'].dtype

dtype('float64')